# Build the whole-bottle classifier dataset

Локальная пересборка датасета входов для bottle-DINO. Если confidence выбранного YOLO-бокса ниже `0.75`, сохраняется **исходное фото без crop**. При confidence `>= 0.75` сохраняется padded crop бутылки. Неоднозначные target-owner случаи остаются только для аудита.

При `OVERWRITE = True` старая директория `datasets/bottle_classifier_crops/` и старый ZIP удаляются непосредственно перед новой генерацией. Исходные 45k изображений не изменяются.

In [ ]:
from pathlib import Path
import json
import subprocess

PROJECT_ROOT = Path('/Users/konstantinmelnikov/Desktop/work/VIscaNEr')
CROP_CONFIDENCE_THRESHOLD = 0.75
BATCH_SIZE = 4
DEVICE = 'auto'  # auto -> MPS on Apple Silicon, CUDA on NVIDIA, otherwise CPU
OVERWRITE = True

PYTHON = PROJECT_ROOT / '.venv' / 'bin' / 'python'
GENERATOR = PROJECT_ROOT / 'build_bottle_classifier_dataset.py'
OUTPUT_ROOT = PROJECT_ROOT / 'datasets' / 'bottle_classifier_crops'
OUTPUT_ZIP = PROJECT_ROOT / 'datasets' / 'bottle_classifier_crops.zip'
required = [
    PYTHON,
    GENERATOR,
    PROJECT_ROOT / 'models' / 'joint_yolo' / 'best.pt',
    PROJECT_ROOT / 'datasets' / 'dinov3_target_crops' / 'crops_metadata.csv',
    PROJECT_ROOT / 'datasets' / 'bottle_images_45k' / 'bottle_images_manifest.csv',
]
missing = [path for path in required if not path.exists()]
assert not missing, f'Missing required inputs: {missing}'
print(f'Project: {PROJECT_ROOT}')
print(f'Policy: confidence < {CROP_CONFIDENCE_THRESHOLD} -> original image; otherwise YOLO crop')
print(f'Will overwrite: {OUTPUT_ROOT} and {OUTPUT_ZIP}')


In [ ]:
command = [
    str(PYTHON),
    '-u',
    str(GENERATOR),
    '--device', DEVICE,
    '--batch-size', str(BATCH_SIZE),
    '--crop-confidence-threshold', str(CROP_CONFIDENCE_THRESHOLD),
]
if OVERWRITE:
    command.append('--overwrite')

print('Starting full regeneration...')
subprocess.run(command, cwd=PROJECT_ROOT, check=True)
print('Generation completed.')


In [ ]:
summary_path = OUTPUT_ROOT / 'build_summary.json'
summary = json.loads(summary_path.read_text(encoding='utf-8'))
print(json.dumps({
    'complete': summary['complete'],
    'rows': summary['rows'],
    'status_counts': summary['status_counts'],
    'image_mode_counts': summary['image_mode_counts'],
    'training_rows': summary['training_rows'],
    'thresholds': summary['thresholds'],
    'archive': summary.get('archive'),
}, ensure_ascii=False, indent=2))
assert summary['complete'], 'Dataset generation did not finish'
assert OUTPUT_ZIP.is_file(), f'Archive was not created: {OUTPUT_ZIP}'
